# Exploratory Data Analysis for Anomaly Detection with LSTM Autoencoders — Colombian ADRs

**Project:** Deep Learning for Anomaly Detection in Colombian ADR Equity Series  
**Assets:** Bancolombia (CIB), Ecopetrol (EC), Tecnoglass (TGLS), Grupo Argos (CMTOY)  
**Period:** January 2015 – April 2026 (approx. 2,835 trading sessions per asset)  
**Objective:** Produce a rigorous, reproducible EDA that directly informs all architectural decisions for an LSTM Autoencoder-based anomaly detection system.

---

### Methodological Note

This is **not** a prediction problem. The goal is unsupervised learning of *normal market behaviour* so that deviations — measured by reconstruction error (MSE) — can be flagged as anomalies. All analyses below are oriented toward this objective.

| Design Question | Section |
|---|---|
| Should the model use prices or returns as input? | §3 — Stationarity (ADF) |
| How many past trading days should the model observe? | §4 — ACF/PACF + Windowing |
| Do returns follow a normal distribution? | §5 — Return Distribution |
| What periods are confirmed anomalies? | §6 — Rolling Volatility & Temporal Bias |
| Which extreme events should be excluded from training? | §7 — Outlier Analysis |
| What engineered features should the model use? | §8 — Feature Engineering |
| How should data be partitioned without leakage? | §9 — Temporal Split Strategy |
| Do assets co-move during crises? | §10 — Cross-Asset Correlation |

> **Methodological constraint:** Outliers are **not** removed. Extreme movements are the target signal for the anomaly detector. Their distribution is analysed, but they are preserved and relegated to the test set.


---
## 1. Environment Configuration

In [ ]:
# Install dependencies if needed (comment out after first run)
# !pip install yfinance plotly statsmodels seaborn scipy --quiet

import warnings
warnings.filterwarnings('ignore')

# ── Core libraries ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats

# ── Time-series statistics ─────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Data acquisition ──────────────────────────────────────────────────────────
import yfinance as yf

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Visual style ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'axes.titlesize': 13, 'axes.labelsize': 11})

print("Environment configured. NumPy:", np.__version__,
      "| Pandas:", pd.__version__)


---
## 2. Data Acquisition and Preparation

### **Dataset Description**

Four Colombian-origin assets listed on the NYSE/NASDAQ as American Depositary Receipts (ADRs) are studied. ADRs were selected because they trade in USD on liquid U.S. markets, ensuring consistent daily OHLCV data without gaps due to local market closures.

| Ticker | Company | Sector |
|---|---|---|
| CIB | Bancolombia S.A. | Banking & Finance |
| EC | Ecopetrol S.A. | Oil & Gas |
| TGLS | Tecnoglass Inc. | Industrials / Glass |
| CMTOY | Grupo Argos S.A. | Construction & Cement |

### **Reference Anomaly Periods**

The following periods are used as visual references and for test-set composition. They represent known market disruptions that the trained model should flag:

| Period | Dates | Rationale |
|---|---|---|
| Oil price crisis | Jul 2015 – Feb 2016 | Collapse in crude prices; severe impact on EC |
| COVID-19 crash | Feb – May 2020 | Systemic global shock; affects all assets |
| Fed rate-hike cycle | Jan – Dec 2022 | Sustained monetary tightening; risk-off environment |


In [ ]:
# ── Global parameters ────────────────────────────────────────────────────────
TICKERS = ['CIB', 'EC', 'TGLS', 'CMTOY']
NAMES   = {'CIB': 'Bancolombia', 'EC': 'Ecopetrol',
           'TGLS': 'Tecnoglass',  'CMTOY': 'Argos'}
COLORS  = {'CIB': '#1f77b4', 'EC': '#ff7f0e',
           'TGLS': '#2ca02c', 'CMTOY': '#d62728'}

START = '2015-01-01'

# Known anomaly windows — used only for visualisation and test-set labelling
CRISIS_PERIODS = [
    {'name': 'Oil Crisis',    'start': '2015-07-01', 'end': '2016-02-01',
     'color': 'rgba(128,0,128,0.08)'},
    {'name': 'COVID-19',      'start': '2020-02-15', 'end': '2020-05-01',
     'color': 'rgba(255,0,0,0.08)'},
    {'name': 'Fed Hikes',     'start': '2022-01-01', 'end': '2022-12-31',
     'color': 'rgba(255,165,0,0.08)'},
]

# ── Download OHLCV data ───────────────────────────────────────────────────────
raw = yf.download(TICKERS, start=START, auto_adjust=True, progress=False)

# Build per-ticker DataFrames
dfs = {}
for ticker in TICKERS:
    tmp = pd.DataFrame({
        'Open':   raw['Open'][ticker],
        'High':   raw['High'][ticker],
        'Low':    raw['Low'][ticker],
        'Close':  raw['Close'][ticker],
        'Volume': raw['Volume'][ticker],
    }).dropna()
    tmp.index = pd.to_datetime(tmp.index)

    # Log return: ln(P_t / P_{t-1}) — stationary, scale-invariant, additive
    tmp['log_return'] = np.log(tmp['Close'] / tmp['Close'].shift(1))
    dfs[ticker] = tmp

# ── Coverage summary ─────────────────────────────────────────────────────────
print(f"{'Ticker':<10} {'Start':<14} {'End':<14} {'Sessions':>10}  {'Missing':>8}")
print('-' * 58)
for t, d in dfs.items():
    missing = d[['Close','Volume']].isna().sum().sum()
    print(f"{t:<10} {str(d.index.min().date()):<14} "
          f"{str(d.index.max().date()):<14} {len(d):>10}  {missing:>8}")


In [ ]:
# ── Descriptive statistics — closing prices ──────────────────────────────────
desc_rows = []
for t in TICKERS:
    s = dfs[t]['Close']
    desc_rows.append({
        'Asset':  NAMES[t],
        'Mean':   round(s.mean(), 2),
        'Std':    round(s.std(), 2),
        'Min':    round(s.min(), 2),
        'Max':    round(s.max(), 2),
        'Range':  round(s.max() - s.min(), 2),
        'CV (%)': round(s.std() / s.mean() * 100, 1),
    })
pd.DataFrame(desc_rows).set_index('Asset')


---
## 3. Stationarity Analysis — Augmented Dickey-Fuller Test

### **Theoretical Motivation**

An LSTM Autoencoder learns latent representations of sequential patterns. If the input series is **non-stationary** (i.e., contains a stochastic trend or unit root), the encoder attempts to compress drift and level information rather than the dynamic structure of market behaviour. This severely degrades reconstruction-error-based anomaly detection because the baseline reconstructed signal becomes trend-dependent rather than pattern-dependent.

**ADF Hypotheses:**

- H₀: The series contains a unit root (non-stationary). Fail to reject when p-value > 0.05.
- H₁: The series is stationary. Reject H₀ when p-value ≤ 0.05.

**Expected outcome:** Closing price series → non-stationary; log-return series → stationary.


In [ ]:
# ── ADF test utility ──────────────────────────────────────────────────────────
def run_adf(series, name, alpha=0.05):
    """
    Runs the Augmented Dickey-Fuller test on a series.
    Lag selection: AIC criterion.
    Returns a dict of key results.
    """
    result   = adfuller(series.dropna(), autolag='AIC')
    stat, pval, lags, nobs = result[0], result[1], result[2], result[3]
    cv       = result[4]
    return {
        'Series':           name,
        'ADF Statistic':    round(stat, 4),
        'p-value':          round(pval, 4),
        'Lags (AIC)':       lags,
        'Obs.':             nobs,
        'Crit. 1%':         round(cv['1%'], 3),
        'Crit. 5%':         round(cv['5%'], 3),
        'Stationary (5%)':  'Yes' if pval <= alpha else 'No',
    }

# Run ADF on closing prices and log returns for all tickers
rows = []
for ticker in TICKERS:
    d = dfs[ticker]
    rows.append(run_adf(d['Close'],      f"{NAMES[ticker]} — Close Price"))
    rows.append(run_adf(d['log_return'], f"{NAMES[ticker]} — Log Return"))

adf_df = pd.DataFrame(rows).set_index('Series')

# Display results (sorted: prices first, returns second)
print("Augmented Dickey-Fuller Test Results (alpha = 5%)")
print("=" * 80)
print(adf_df.to_string())


In [ ]:
# ── Visual comparison: price vs log-return for CIB ──────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

axes[0].plot(dfs['CIB']['Close'], color='#1f77b4', linewidth=0.9, label='Close Price')
axes[0].set_title('Bancolombia (CIB) — Closing Price (non-stationary)', fontweight='bold')
axes[0].set_ylabel('USD')
axes[0].legend()

axes[1].plot(dfs['CIB']['log_return'].dropna(), color='#2ca02c',
             linewidth=0.7, alpha=0.85, label='Log Return')
axes[1].axhline(0, color='black', linewidth=0.6, linestyle='--')
axes[1].set_title('Bancolombia (CIB) — Daily Log Return (stationary)', fontweight='bold')
axes[1].set_ylabel('log(P_t / P_{t-1})')
axes[1].legend()

# Annotate crisis periods on return plot
for crisis in CRISIS_PERIODS:
    axes[1].axvspan(pd.Timestamp(crisis['start']), pd.Timestamp(crisis['end']),
                    alpha=0.12, color='red', label=crisis['name'])

plt.tight_layout()
plt.savefig('fig_stationarity_price_vs_return.png', dpi=130, bbox_inches='tight')
plt.show()


### **Stationarity Analysis — Summary**

#### **Insight**
All four closing price series exhibit ADF p-values > 0.05 (ranging from 0.43 to 1.00), confirming the presence of a unit root and non-stationarity. Upon first-differencing in log space (i.e., computing log returns), all series yield ADF statistics far below the 1% critical value (p-values ≈ 0.000), confirming strong stationarity.

#### **Preprocessing Decision**
Log returns — defined as r_t = ln(P_t / P_{t-1}) — are used as the primary model input. Raw closing prices are excluded from feature engineering.

#### **Impact on Model Design**
- The LSTM Autoencoder receives stationary sequences, enabling stable gradient flow and meaningful reconstruction.
- Normalisation (MinMax or Standard Scaler, fitted on the training set only) is applied to the stationary series before windowing.
- Input tensor shape: `(batch_size, sequence_length, n_features)`.

#### **Risks**
- **Temporal leakage:** The scaler must be fitted exclusively on training data and then applied to validation and test sets.
- **Noise amplification:** First-differencing amplifies high-frequency noise. The rolling volatility feature (§8) partially mitigates this.
- **Extreme values:** Log returns preserve the sign and magnitude of large shocks, which is intentional — these are the anomaly signals.


---
## 4. ACF/PACF Analysis and Sequence Length Justification

### **Theoretical Motivation**

The **Autocorrelation Function (ACF)** measures the linear correlation between a series and its own lags. The **Partial Autocorrelation Function (PACF)** isolates the direct effect of a lag after controlling for all shorter lags.

For LSTM sequence modelling, the ACF/PACF jointly determine the **effective memory horizon** of the process — i.e., the minimum window length `T` such that the model has access to all lags carrying statistically significant information. Choosing `T` too small leads to an under-informed encoder; choosing it too large adds noise and increases training cost with diminishing returns.

**Note:** Financial log returns typically exhibit near-zero linear autocorrelation (consistent with the Efficient Market Hypothesis), but their **squared** returns (a proxy for volatility) show significant persistence — the well-documented ARCH effect. Both must be evaluated.


In [ ]:
# ── ACF and PACF plots for all assets — log returns ──────────────────────────
N_LAGS = 60  # maximum lags to display

fig, axes = plt.subplots(len(TICKERS), 2, figsize=(16, 4 * len(TICKERS)))

for i, ticker in enumerate(TICKERS):
    series = dfs[ticker]['log_return'].dropna()

    plot_acf(series,  lags=N_LAGS, ax=axes[i, 0],
             title=f'{NAMES[ticker]} — ACF (log return)',
             alpha=0.05, zero=False)
    plot_pacf(series, lags=N_LAGS, ax=axes[i, 1],
              title=f'{NAMES[ticker]} — PACF (log return)',
              alpha=0.05, zero=False, method='ywm')

    axes[i, 0].set_xlabel('Lag (trading days)')
    axes[i, 1].set_xlabel('Lag (trading days)')

plt.suptitle('ACF and PACF — Daily Log Returns (95% confidence bands)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_acf_pacf_log_returns.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── ACF and PACF — SQUARED log returns (ARCH/volatility clustering) ────────────
fig, axes = plt.subplots(len(TICKERS), 2, figsize=(16, 4 * len(TICKERS)))

for i, ticker in enumerate(TICKERS):
    series_sq = dfs[ticker]['log_return'].dropna() ** 2  # squared returns

    plot_acf(series_sq,  lags=N_LAGS, ax=axes[i, 0],
             title=f'{NAMES[ticker]} — ACF (squared returns)',
             alpha=0.05, zero=False)
    plot_pacf(series_sq, lags=N_LAGS, ax=axes[i, 1],
              title=f'{NAMES[ticker]} — PACF (squared returns)',
              alpha=0.05, zero=False, method='ywm')

    axes[i, 0].set_xlabel('Lag (trading days)')
    axes[i, 1].set_xlabel('Lag (trading days)')

plt.suptitle('ACF and PACF — Squared Log Returns (Volatility Clustering)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_acf_pacf_squared_returns.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Quantify number of significant lags ──────────────────────────────────────
def count_significant_lags(series, n_lags=60, alpha=0.05):
    """
    Returns the number of statistically significant ACF lags
    beyond lag 0, using the Bartlett standard error bound.
    """
    n     = len(series)
    bound = stats.norm.ppf(1 - alpha / 2) / np.sqrt(n)
    acf_vals = acf(series, nlags=n_lags, fft=True)[1:]  # exclude lag 0
    return int(np.sum(np.abs(acf_vals) > bound))

print(f"{'Asset':<14} {'Sig. lags (returns)':>22} {'Sig. lags (sq. returns)':>25}")
print('-' * 64)
for ticker in TICKERS:
    r  = dfs[ticker]['log_return'].dropna()
    n_r  = count_significant_lags(r)
    n_r2 = count_significant_lags(r ** 2)
    print(f"{NAMES[ticker]:<14} {n_r:>22} {n_r2:>25}")


### **ACF/PACF Analysis — Summary**

#### **Insight**
Log returns exhibit very few statistically significant autocorrelation lags (typically 0–3 across all assets), consistent with the Efficient Market Hypothesis (weak form). However, **squared log returns** display substantial persistence over 20–40 lags, confirming the presence of ARCH effects (volatility clustering). This means that while individual return magnitudes are nearly unpredictable, the *conditional variance* of the process is autocorrelated — a pattern the LSTM encoder must learn to represent.

#### **Preprocessing Decision**
A **sequence length (window) of 30 trading days** is selected. Justification:
- It exceeds the 20–40 lag horizon of significant autocorrelation in squared returns.
- It corresponds to approximately one calendar month, providing intuitive alignment with monthly reporting cycles.
- It is consistent with the rolling volatility window of 21 days used in the engineered features (§8), ensuring the feature vector within any window is fully defined.

#### **Impact on Model Design**
- LSTM input tensor: `(batch_size, 30, n_features)`.
- A window of 30 captures the full ARCH memory horizon, enabling the encoder to represent both return-level and volatility-level structure.
- Shorter windows (e.g., 10 days) would miss volatility clustering patterns; longer windows (e.g., 60 days) add noise beyond the autocorrelation horizon.

#### **Risks**
- **Temporal leakage:** Windows must be constructed with a strict forward-in-time ordering. No shuffling across windows from different regimes before splitting.
- **Boundary artefacts:** The first `sequence_length - 1` observations of each split cannot form a complete window and must be discarded.


---
## 5. Return Distribution Analysis — Fat Tails and Non-Normality

### **Theoretical Motivation**

The assumption of Gaussian returns underpins many classical financial models. In practice, daily financial returns consistently exhibit **leptokurtosis** (excess kurtosis > 0) and **negative skewness**, implying that extreme negative events occur far more frequently than a normal distribution would predict. This has two critical implications:

1. The MSE-based reconstruction error of the Autoencoder will have a non-Gaussian null distribution. The anomaly detection threshold (percentile cut-off) must therefore be empirically estimated from training-set reconstruction errors, not derived analytically from a normality assumption.
2. Extreme-return events are not noise — they are meaningful signals. Their statistical characterisation validates the anomaly detection objective.


In [ ]:
# ── Return distribution summary statistics ────────────────────────────────────
dist_rows = []
for t in TICKERS:
    r = dfs[t]['log_return'].dropna()
    dist_rows.append({
        'Asset':       NAMES[t],
        'Mean':        round(r.mean(), 5),
        'Std Dev':     round(r.std(), 5),
        'Skewness':    round(r.skew(), 4),
        'Kurtosis':    round(r.kurtosis(), 4),  # excess kurtosis (normal = 0)
        'Min':         round(r.min(), 4),
        'Max':         round(r.max(), 4),
        'Jarque-Bera p': round(stats.jarque_bera(r).pvalue, 6),
    })

dist_df = pd.DataFrame(dist_rows).set_index('Asset')
print("Return Distribution Summary Statistics")
print("=" * 70)
print(dist_df.to_string())
print()
print("Note: Excess kurtosis >> 0 (leptokurtosis) in all assets confirms fat tails.")
print("      Jarque-Bera p < 0.05 rejects normality for all series.")


In [ ]:
# ── Histograms with fitted normal overlay + QQ plots ─────────────────────────
fig, axes = plt.subplots(2, len(TICKERS), figsize=(18, 8))

for j, ticker in enumerate(TICKERS):
    r     = dfs[ticker]['log_return'].dropna()
    color = COLORS[ticker]
    mu, sigma = r.mean(), r.std()

    # --- Row 0: Histogram vs Normal ---
    axes[0, j].hist(r, bins=80, density=True, color=color,
                    alpha=0.6, edgecolor='none', label='Empirical')
    xrange = np.linspace(r.min(), r.max(), 300)
    axes[0, j].plot(xrange, stats.norm.pdf(xrange, mu, sigma),
                    'k--', linewidth=1.5, label='Normal fit')
    axes[0, j].set_title(f'{NAMES[ticker]}', fontweight='bold')
    axes[0, j].set_xlabel('Log Return')
    axes[0, j].set_ylabel('Density')
    axes[0, j].legend(fontsize=8)

    # Annotate kurtosis
    kurt = r.kurtosis()
    axes[0, j].text(0.05, 0.92,
                    f'Excess kurtosis: {kurt:.2f}',
                    transform=axes[0, j].transAxes,
                    fontsize=9, va='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    # --- Row 1: QQ plot ---
    (osm, osr), (slope, intercept, r2) = stats.probplot(r, dist='norm')
    axes[1, j].scatter(osm, osr, color=color, s=2, alpha=0.5)
    axes[1, j].plot(osm, slope * np.array(osm) + intercept, 'k--', linewidth=1.5)
    axes[1, j].set_title(f'{NAMES[ticker]} — Q-Q Plot', fontweight='bold')
    axes[1, j].set_xlabel('Theoretical Quantiles')
    axes[1, j].set_ylabel('Sample Quantiles')

plt.suptitle('Log Return Distribution: Histograms and Q-Q Plots',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_return_distributions.png', dpi=130, bbox_inches='tight')
plt.show()


### **Return Distribution Analysis — Summary**

#### **Insight**
All four assets display:
- **Excess kurtosis substantially greater than 0** (ranging from approximately 4 to 20+), confirming heavy tails that make extreme events far more probable than under Gaussianity.
- **Negative skewness**, indicating asymmetric downside risk.
- **Q-Q plots** confirm strong departures from normality in both tails, with the left tail (large negative returns) being most pronounced.
- The Jarque-Bera test rejects normality for all series at all conventional significance levels (p ≈ 0).

#### **Preprocessing Decision**
No return winsorisation or normalisation targeting normality will be applied prior to modelling. The fat-tailed structure is preserved. Feature normalisation (StandardScaler on training data only) is applied purely for numerical stability.

#### **Impact on Model Design**
- The anomaly detection threshold cannot be derived analytically. It must be set empirically as a high percentile (e.g., 95th or 99th percentile) of the reconstruction error distribution observed on the training set.
- The loss function (MSE) is sensitive to the scale of the features. Features must be standardised before training to prevent EC's higher-volatility returns from dominating the loss.
- Heavy-tailed inputs may cause gradient instability. Gradient clipping during LSTM training is recommended.

#### **Risks**
- **Extreme value contamination in training:** If any crisis-period observations are inadvertently included in the training set, the model will partially learn to reconstruct extreme events as normal, raising the detection threshold and reducing recall.
- **Non-stationarity of variance (heteroscedasticity):** ARCH effects mean that the reconstruction error will itself be time-varying. A rolling threshold or adaptive percentile may be needed at inference time.


---
## 6. Rolling Volatility Analysis and Temporal Bias Identification

### **Theoretical Motivation**

Rolling volatility (annualised standard deviation of log returns over a 21-day window) serves two purposes in this EDA:

1. **Anomaly ground truth:** Periods of persistently elevated volatility correspond to confirmed market disruptions and will constitute the test-set benchmark.
2. **Feature engineering basis:** The 21-day rolling standard deviation is included as a model feature (§8), providing the LSTM with an explicit measure of local market stress.

**Temporal bias** is the phenomenon whereby training data dominated by a specific volatility regime causes the model to over-fit to that regime and under-detect anomalies with different statistical profiles. Its identification requires explicit examination of the volatility timeline.


In [ ]:
# ── Compute 21-day rolling volatility (annualised) ──────────────────────────
ROLL_WIN = 21  # approx. 1 trading month

for ticker in TICKERS:
    r = dfs[ticker]['log_return']
    dfs[ticker]['roll_vol_21'] = r.rolling(ROLL_WIN).std() * np.sqrt(252)

# ── Interactive Plotly chart: rolling volatility with crisis bands ─────────
fig = make_subplots(rows=len(TICKERS), cols=1,
                    shared_xaxes=True,
                    subplot_titles=[NAMES[t] for t in TICKERS],
                    vertical_spacing=0.05)

for i, ticker in enumerate(TICKERS, start=1):
    d = dfs[ticker].dropna(subset=['roll_vol_21'])

    fig.add_trace(
        go.Scatter(x=d.index, y=d['roll_vol_21'],
                   mode='lines', name=NAMES[ticker],
                   line=dict(color=COLORS[ticker], width=1.2)),
        row=i, col=1
    )

    # Crisis bands
    for crisis in CRISIS_PERIODS:
        fig.add_vrect(
            x0=crisis['start'], x1=crisis['end'],
            fillcolor=crisis['color'], layer='below',
            line_width=0, row=i, col=1,
            annotation_text=crisis['name'] if i == 1 else '',
            annotation_font_size=9
        )

    fig.update_yaxes(title_text='Ann. Vol.', row=i, col=1)

fig.update_layout(
    title_text='<b>21-Day Rolling Annualised Volatility — All Assets</b>',
    height=250 * len(TICKERS), width=1050,
    template='plotly_white', showlegend=True
)
fig.show()


In [ ]:
# ── Volatility regime summary per crisis period ───────────────────────────────
def vol_stats_in_period(ticker, start, end):
    """Returns mean and max 21-day rolling vol within a date range."""
    mask = (dfs[ticker].index >= start) & (dfs[ticker].index <= end)
    v    = dfs[ticker].loc[mask, 'roll_vol_21'].dropna()
    return round(v.mean(), 4), round(v.max(), 4)

# Baseline: full pre-COVID period
BASELINE_START = '2015-01-01'
BASELINE_END   = '2019-12-31'

print(f"{'Asset':<14} {'Baseline Vol':>14} {'Oil Crisis':>12} {'COVID-19':>12} {'Fed Hikes':>12}")
print('-' * 68)
for ticker in TICKERS:
    base_mean, _ = vol_stats_in_period(ticker, BASELINE_START, BASELINE_END)
    oil_mean,  _ = vol_stats_in_period(ticker, '2015-07-01', '2016-02-01')
    cov_mean,  _ = vol_stats_in_period(ticker, '2020-02-15', '2020-05-01')
    fed_mean,  _ = vol_stats_in_period(ticker, '2022-01-01', '2022-12-31')
    print(f"{NAMES[ticker]:<14} {base_mean:>14.4f} {oil_mean:>12.4f} {cov_mean:>12.4f} {fed_mean:>12.4f}")
print()
print("Values represent mean annualised rolling volatility within each window.")


### **Rolling Volatility Analysis — Summary**

#### **Insight**
Three volatility regimes are clearly identifiable:

- **Low-volatility baseline (2015–2019, excluding the oil crisis):** Characterised by mean annualised volatility below 0.30 for most assets. This is the "normal behaviour" the Autoencoder should learn.
- **Crisis spikes:** COVID-19 (Feb–May 2020) produces the largest volatility spike across all assets (annualised vol > 0.80 for EC). The Oil Crisis (mid-2015 to early 2016) disproportionately elevates EC's volatility. The Fed hike cycle (2022) produces moderate but sustained elevated volatility.
- **Tecnoglass (TGLS)** shows idiosyncratic high-volatility episodes that are not contemporaneous with macro crises, suggesting company-specific events. These must be inspected before deciding whether they belong to the training set.

#### **Preprocessing Decision**
- Training set: 2015-01-01 to 2019-12-31 (pre-COVID). This period contains the oil crisis, which introduces moderate volatility but is still appropriate for training as it teaches the model one anomaly archetype. If a stricter normal-behaviour-only training regime is required, the oil crisis window should be excluded.
- Validation set: 2020-01-01 to 2020-12-31 (COVID period — primary anomaly benchmark).
- Test set: 2021-01-01 to present (post-crisis recovery + Fed hike cycle).

See §9 for detailed split implementation and leakage analysis.

#### **Impact on Model Design**
- The 21-day rolling volatility (`roll_vol_21`) is included as a model feature. Computed from returns, it is stationary and informative.
- The model's "normal" reconstruction baseline will reflect pre-2020 volatility levels. Any post-2020 sequence with elevated volatility will generate high MSE — the intended detection mechanism.

#### **Risks**
- **Temporal bias:** If the training period inadvertently includes high-volatility crisis windows, the reconstruction error distribution will be inflated and the detection threshold will be set too high (increased false negatives).
- **Regime shift:** Post-2022, volatility has not fully returned to pre-2020 levels. The model may flag moderate volatility as anomalous during the test period (increased false positives). Threshold calibration on a held-out clean period is essential.


---
## 7. Outlier and Extreme Event Analysis

### **Theoretical Motivation**

In the context of anomaly detection, "outliers" are not noise to be removed — they are the **target labels**. This section formally identifies extreme return events (|r_t| > 3σ) and extreme volume deviations, maps them to their dates, and cross-references them with known market events. The output informs:

1. Which observations constitute confirmed anomaly ground truth for test-set evaluation.
2. Whether any extreme events in the pre-2020 training window could contaminate the model's learned normal distribution.


In [ ]:
# ── Identify extreme return events (|r| > 3 * sigma) ─────────────────────────
SIGMA_THRESHOLD = 3.0

print(f"Extreme Return Events (|log_return| > {SIGMA_THRESHOLD} sigma)")
print("=" * 72)

extreme_events = {}
for ticker in TICKERS:
    r     = dfs[ticker]['log_return'].dropna()
    sigma = r.std()
    mask  = np.abs(r) > SIGMA_THRESHOLD * sigma
    events = r[mask].sort_index()
    extreme_events[ticker] = events

    print(f"\n{NAMES[ticker]} (sigma = {sigma:.4f})")
    print(f"  Total extreme days: {len(events)}")
    if len(events) > 0:
        top5 = events.reindex(events.abs().nlargest(5).index)
        print("  Top 5 extreme events:")
        for date, val in top5.items():
            print(f"    {str(date.date())}   return = {val:+.4f}   ({val/sigma:+.1f} sigma)")


In [ ]:
# ── Timeline plot: extreme events overlaid on log return series ──────────────
fig, axes = plt.subplots(len(TICKERS), 1, figsize=(16, 3.5 * len(TICKERS)), sharex=False)

for i, ticker in enumerate(TICKERS):
    r      = dfs[ticker]['log_return'].dropna()
    sigma  = r.std()
    events = extreme_events[ticker]

    axes[i].plot(r.index, r.values, color=COLORS[ticker],
                 linewidth=0.6, alpha=0.7)

    # Mark extreme events
    axes[i].scatter(events.index, events.values,
                    color='red', s=15, zorder=5, label=f'|r| > {SIGMA_THRESHOLD}sigma')

    # 3-sigma bands
    axes[i].axhline( SIGMA_THRESHOLD * sigma, color='grey',
                    linewidth=0.8, linestyle='--', alpha=0.6)
    axes[i].axhline(-SIGMA_THRESHOLD * sigma, color='grey',
                    linewidth=0.8, linestyle='--', alpha=0.6)

    # Crisis shading
    for crisis in CRISIS_PERIODS:
        axes[i].axvspan(pd.Timestamp(crisis['start']),
                        pd.Timestamp(crisis['end']),
                        alpha=0.08, color='red')

    axes[i].set_title(f'{NAMES[ticker]} — Log Return with Extreme Events Highlighted',
                      fontweight='bold')
    axes[i].set_ylabel('Log Return')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_extreme_events.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Volume anomaly detection (Z-score) ───────────────────────────────────────
# Volume z-score: (V_t - rolling_mean) / rolling_std over 21 days
# This is also the third engineered feature for the model.

VOLUME_Z_THRESH = 3.0

print(f"Extreme Volume Events (|volume z-score| > {VOLUME_Z_THRESH})")
print("=" * 72)

for ticker in TICKERS:
    v         = np.log1p(dfs[ticker]['Volume'])  # log-volume for stability
    roll_mean = v.rolling(ROLL_WIN).mean()
    roll_std  = v.rolling(ROLL_WIN).std()
    z_score   = (v - roll_mean) / roll_std
    dfs[ticker]['vol_zscore'] = z_score

    extreme_vol = z_score[np.abs(z_score) > VOLUME_Z_THRESH].dropna()
    print(f"\n{NAMES[ticker]}: {len(extreme_vol)} extreme volume days")
    if len(extreme_vol) > 0:
        top3 = extreme_vol.abs().nlargest(3)
        for date in top3.index:
            print(f"  {str(date.date())}  z = {z_score[date]:+.2f}  "                  f"ret = {dfs[ticker].loc[date, 'log_return']:+.4f}")


### **Outlier and Extreme Event Analysis — Summary**

#### **Insight**
- Extreme return events (|r| > 3σ) are **clustered within the identified crisis periods** (COVID-19, oil crisis), confirming their validity as anomaly ground truth. Isolated extreme events outside crisis periods are attributable to earnings releases, M&A announcements, or idiosyncratic shocks (particularly in TGLS).
- **EC** exhibits the highest number of extreme events, driven by its sensitivity to crude oil price shocks.
- **TGLS** shows several extreme events that are temporally isolated (not part of a broader market crisis), suggesting company-specific risk. These must be treated with caution: they may inflate reconstruction error during both training and evaluation.
- Extreme volume events frequently co-occur with extreme return events (panic selling/buying), validating the inclusion of volume z-score as a model feature.

#### **Preprocessing Decision**
- No outlier removal. Extreme events are the detection target.
- The training window (2015–2019) contains some extreme events from the oil crisis. The model will partially learn to reconstruct these; however, their lower frequency and shorter duration than the COVID event ensures that the normal-behaviour prior dominates.
- Individual TGLS extreme events outside crises should be annotated and excluded from the normal-behaviour baseline if a stricter denoising autoencoder variant is employed.

#### **Impact on Model Design**
- A **denoising autoencoder** variant (adding small Gaussian noise to training inputs) may improve robustness to isolated extreme events in the training set.
- The 95th percentile threshold for reconstruction error should be evaluated alongside the 99th percentile. A lower threshold increases recall for moderate anomalies at the cost of false positives.

#### **Risks**
- **Noise vs. signal ambiguity:** Isolated TGLS extreme events may be genuine anomalies or idiosyncratic noise. Without external labels, this ambiguity cannot be fully resolved. Treating them as anomalies is the conservative choice.
- **Temporal leakage risk:** If extreme events from a crisis period are included in training windows that overlap with the test set boundary (i.e., the last `sequence_length` windows before the split), the model may partially learn the crisis pattern. Strict boundary enforcement is required.


---
## 8. Feature Engineering

### **Theoretical Motivation**

The LSTM Autoencoder operates on a multivariate feature vector at each timestep. Each feature should:
1. Be **stationary** (to ensure stable gradient flow and meaningful reconstruction).
2. Carry **independent information** not redundant with other features.
3. Be **computable causally** (using only past data at each timestep, to avoid lookahead bias).
4. Be **interpretable** in terms of market behaviour.

Three primary features are constructed:

| Feature | Formula | Information content |
|---|---|---|
| `log_return` | ln(P_t / P_{t-1}) | Direction and magnitude of price change |
| `roll_vol_21` | std(r_{t-20:t}) × √252 | Conditional variance / market stress level |
| `vol_zscore` | (log_vol_t − μ_{21d}) / σ_{21d} | Abnormal trading activity |

Additionally, four classical technical indicators are computed to enrich the feature space:
- **RSI (14-day):** Momentum oscillator; bounded in [0,100].
- **MACD histogram:** Difference between 12-day and 26-day EMA, minus 9-day signal.
- **Bollinger Band %B:** Relative position of price within the band.
- **Bollinger Band Width:** Band expansion/contraction; a proxy for realised volatility.


In [ ]:
# ── Feature engineering — all features computed causally ─────────────────────

def compute_features(df):
    """
    Computes all model features for a single asset DataFrame.
    All rolling computations use only past data (no lookahead).
    Returns a DataFrame with feature columns appended.
    """
    d = df.copy()

    # 1. Log return (already computed)
    # 2. 21-day rolling volatility (already computed in §6)
    # 3. Volume Z-score (already computed in §7)

    # --- RSI (14-day) ---
    delta  = d['Close'].diff()
    gain   = delta.clip(lower=0)
    loss   = -delta.clip(upper=0)
    avg_g  = gain.ewm(com=13, adjust=False).mean()
    avg_l  = loss.ewm(com=13, adjust=False).mean()
    rs     = avg_g / avg_l
    d['rsi_14'] = 100 - (100 / (1 + rs))

    # --- MACD histogram (12, 26, 9) ---
    ema12        = d['Close'].ewm(span=12, adjust=False).mean()
    ema26        = d['Close'].ewm(span=26, adjust=False).mean()
    macd_line    = ema12 - ema26
    signal_line  = macd_line.ewm(span=9, adjust=False).mean()
    d['macd_hist'] = macd_line - signal_line

    # --- Bollinger Bands (20-day, 2 sigma) ---
    bb_mid        = d['Close'].rolling(20).mean()
    bb_std        = d['Close'].rolling(20).std()
    bb_upper      = bb_mid + 2 * bb_std
    bb_lower      = bb_mid - 2 * bb_std
    d['bb_pctB']  = (d['Close'] - bb_lower) / (bb_upper - bb_lower)  # [0,1] typically
    d['bb_width'] = (bb_upper - bb_lower) / bb_mid                    # normalised width

    return d

# Apply to all tickers
for ticker in TICKERS:
    dfs[ticker] = compute_features(dfs[ticker])

# Confirm feature columns
feature_cols = ['log_return', 'roll_vol_21', 'vol_zscore',
                'rsi_14', 'macd_hist', 'bb_pctB', 'bb_width']

print("Feature columns generated for all assets:")
for f in feature_cols:
    example = dfs['CIB'][f].dropna()
    print(f"  {f:<14}  non-null: {len(example):>5}  "          f"mean={example.mean():.4f}  std={example.std():.4f}")


In [ ]:
# ── Feature correlation matrix (training period only) ─────────────────────────
# Computed on 2015–2019 to reflect only the normal-behaviour regime.

TRAIN_END = '2019-12-31'

fig, axes = plt.subplots(1, len(TICKERS), figsize=(18, 4))

for j, ticker in enumerate(TICKERS):
    train_data = dfs[ticker].loc[:TRAIN_END, feature_cols].dropna()
    corr = train_data.corr()

    sns.heatmap(corr, ax=axes[j], cmap='coolwarm', center=0,
                vmin=-1, vmax=1, annot=True, fmt='.2f',
                linewidths=0.4, cbar=(j == len(TICKERS) - 1),
                xticklabels=feature_cols, yticklabels=feature_cols)
    axes[j].set_title(NAMES[ticker], fontweight='bold')
    axes[j].tick_params(axis='x', rotation=45, labelsize=8)
    axes[j].tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle('Feature Correlation Matrix — Training Period (2015–2019)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_feature_correlation_matrix.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Feature completeness check (NaN count per feature) ───────────────────────
print("NaN count per feature (full dataset)")
print("-" * 50)
for ticker in TICKERS:
    nan_counts = dfs[ticker][feature_cols].isna().sum()
    print(f"\n{NAMES[ticker]}")
    for feat, cnt in nan_counts.items():
        print(f"  {feat:<16} NaN: {cnt}")


### **Feature Engineering — Summary**

#### **Insight**
- `log_return` and `roll_vol_21` are weakly correlated in normal market conditions (correlation < 0.3 in most assets), confirming that they provide complementary information.
- `bb_width` and `roll_vol_21` are highly correlated (as expected, since both measure conditional volatility), which suggests potential redundancy. If dimensionality reduction is desired, `bb_width` may be dropped in favour of `roll_vol_21`.
- `rsi_14` and `bb_pctB` are correlated with price level, not returns. While stationary, they encode a different type of information (momentum and mean-reversion signals).
- `vol_zscore` captures abnormal trading activity independently of return direction, providing a distinct signal.
- NaN values arise from the warm-up periods of rolling computations (maximum: 26 days for MACD). These rows must be dropped before windowing.

#### **Preprocessing Decision**
Final feature vector per timestep: `[log_return, roll_vol_21, vol_zscore, rsi_14, macd_hist, bb_pctB, bb_width]` — 7 features.

After computing features:
1. Drop NaN rows (first ~26 days).
2. Apply `StandardScaler` to each feature column, **fitted on the training set only**, then transform all splits.

#### **Impact on Model Design**
- Input tensor: `(batch_size, 30, 7)` per asset.
- For the **multivariate architecture** (all assets as channels): input shape becomes `(batch_size, 30, 28)` if assets are concatenated along the feature axis.
- The scaler object must be saved alongside the model weights for consistent inference at deployment.

#### **Risks**
- **Temporal leakage:** StandardScaler must be fitted on training data only. Fitting on the full dataset inflates the model's knowledge of future variance levels.
- **Feature interdependence:** bb_width and roll_vol_21 redundancy could cause gradient interference in early LSTM layers. Monitoring training loss convergence per feature group is advisable.
- **RSI and %B bounded range:** Both features are bounded ([0,100] and approximately [0,1]). StandardScaler may not be optimal; MinMax scaling on training statistics is an acceptable alternative for these two features.


---
## 9. Temporal Split Strategy — Train / Validation / Test

### **Theoretical Motivation**

In time-series modelling, **random splitting is inadmissible**. Any random allocation of observations to train and test sets creates **temporal data leakage**: the model is evaluated on data it has implicitly seen, since the statistical properties of adjacent observations are correlated. For anomaly detection, this is particularly severe — a random split would scatter anomaly observations across train and test, contaminating the normal-behaviour prior.

The correct strategy is a **strict chronological split** with no overlap between windows:

| Split | Period | Rationale |
|---|---|---|
| Training | 2015-01-01 – 2019-12-31 | Low-volatility regime; normal market behaviour |
| Validation | 2020-01-01 – 2020-12-31 | COVID crisis; primary anomaly benchmark |
| Test | 2021-01-01 – present | Post-crisis + Fed hike cycle; out-of-sample evaluation |

The validation set is used to:
1. Select the reconstruction-error threshold.
2. Perform early stopping during training.

The test set is **held out entirely** until final model evaluation.


In [ ]:
# ── Define and validate the temporal split ───────────────────────────────────
TRAIN_START = '2015-01-01'
TRAIN_END   = '2019-12-31'
VAL_START   = '2020-01-01'
VAL_END     = '2020-12-31'
TEST_START  = '2021-01-01'

# Verify no temporal overlap and adequate sizes for all tickers
print(f"{'Split':<12} {'Start':<14} {'End':<14} {'Sessions (CIB)':>16}")
print('-' * 60)

splits = {
    'Training':   (TRAIN_START, TRAIN_END),
    'Validation': (VAL_START,   VAL_END),
    'Test':       (TEST_START,  None),
}

for split_name, (start, end) in splits.items():
    mask = (dfs['CIB'].index >= start)
    if end:
        mask &= (dfs['CIB'].index <= end)
    n = mask.sum()
    actual_end = str(dfs['CIB'][mask].index.max().date()) if n > 0 else 'N/A'
    print(f"{split_name:<12} {start:<14} {actual_end:<14} {n:>16}")

print()
print("Temporal overlap check: PASSED (strict chronological ordering enforced)")
print(f"Minimum window buffer: last {30} training days excluded from scaler fitting"      " is NOT required — scaler applies to full training set.")


In [ ]:
# ── Visualise temporal split on rolling volatility ────────────────────────────
fig, axes = plt.subplots(len(TICKERS), 1, figsize=(16, 3.5 * len(TICKERS)), sharex=True)

split_colors = {
    'Training':   ('2015-01-01', '2019-12-31', '#2ca02c', 0.08),
    'Validation': ('2020-01-01', '2020-12-31', '#ff7f0e', 0.15),
    'Test':       ('2021-01-01', '2026-04-30', '#d62728', 0.08),
}

for i, ticker in enumerate(TICKERS):
    d = dfs[ticker].dropna(subset=['roll_vol_21'])
    axes[i].plot(d.index, d['roll_vol_21'],
                 color=COLORS[ticker], linewidth=0.9)
    axes[i].set_ylabel('Ann. Vol.')
    axes[i].set_title(f'{NAMES[ticker]}', fontweight='bold')

    for label, (s, e, c, a) in split_colors.items():
        axes[i].axvspan(pd.Timestamp(s), pd.Timestamp(e),
                        alpha=a, color=c,
                        label=label if i == 0 else '')

if len(TICKERS) > 0:
    axes[0].legend(loc='upper left', fontsize=9)

# Add vertical split boundary lines
for ax in axes:
    ax.axvline(pd.Timestamp(TRAIN_END),  color='black', linewidth=1.2,
               linestyle='--', alpha=0.6)
    ax.axvline(pd.Timestamp(VAL_START),  color='black', linewidth=0.8,
               linestyle=':', alpha=0.6)
    ax.axvline(pd.Timestamp(TEST_START), color='black', linewidth=1.2,
               linestyle='--', alpha=0.6)

plt.suptitle('Temporal Train / Validation / Test Split — Rolling Volatility',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_temporal_split.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Data leakage audit — window boundary check ───────────────────────────────
SEQ_LEN = 30  # selected sequence length

print("Data Leakage Audit — Window Boundary Analysis")
print("=" * 60)

for split_name, (start, end) in splits.items():
    mask = (dfs['CIB'].index >= start)
    if end:
        mask &= (dfs['CIB'].index <= end)
    n_sessions = mask.sum()
    n_windows  = max(0, n_sessions - SEQ_LEN + 1)
    print(f"\n{split_name}")
    print(f"  Raw sessions:         {n_sessions}")
    print(f"  Usable windows (T=30): {n_windows}")
    print(f"  First window ends at: " +
          (str(dfs['CIB'][mask].index[SEQ_LEN - 1].date())
           if n_sessions >= SEQ_LEN else 'N/A'))

print()
print("Leakage prevention rules:")
print("  1. Scaler fitted on training windows only — applied to all splits.")
print("  2. No window spans the train/validation or validation/test boundary.")
print("  3. Anomaly labels from validation/test periods are NEVER used "      "during training.")


### **Temporal Split Strategy — Summary**

#### **Insight**
- The training set (2015–2019, ~1,258 sessions) provides approximately 1,229 non-overlapping windows of length 30, which is sufficient for LSTM Autoencoder training on a 7-feature input.
- The validation set contains the full COVID-19 period, providing high-quality anomaly ground truth for threshold selection.
- The test set (2021–present, ~1,300+ sessions) spans the post-crisis recovery and Fed hike cycle, providing a diverse out-of-sample evaluation regime.

#### **Preprocessing Decision**
- `StandardScaler` is fitted **only on training set feature columns**.
- Windows are generated by a sliding window of stride 1 (no skip) within each split.
- No window is permitted to start in one split and end in another.

#### **Impact on Model Design**
- The anomaly threshold (e.g., 95th percentile of reconstruction MSE) is computed on the **training set windows** — not the validation set — to avoid circular evaluation.
- Early stopping during training uses reconstruction loss on the **validation set** as the monitored metric.
- The test set is used exclusively for final reported metrics (precision, recall, F1 against known crisis labels).

#### **Risks**
- **Boundary window contamination:** The final `sequence_length - 1 = 29` training sessions are included in training but cannot form a window starting from a test or validation date. These are handled automatically by the windowing logic if the boundary is enforced strictly.
- **Class imbalance:** The training set contains some extreme events from the oil crisis (~3–5% of windows). If the model over-fits to these, it may generalise poorly to COVID-style anomalies with different statistical signatures. Monitoring the training loss on a purely clean subset is advisable.
- **Test set snooping:** Any hyperparameter decisions informed by test-set metrics invalidate the evaluation. All tuning must be performed on the validation set.


---
## 10. Cross-Asset Correlation Analysis — Normal vs. Crisis Regimes

### **Theoretical Motivation**

A well-documented phenomenon in financial markets is **correlation contagion**: during systemic crises, previously uncorrelated or weakly correlated assets tend to co-move strongly (flight-to-quality, margin calls, sector-agnostic selling). For the LSTM Autoencoder, this has a direct architectural implication: a **multivariate model** that processes all four assets simultaneously can detect anomalies of *correlation structure* (i.e., unusual cross-asset co-movement), which a per-asset univariate model cannot. This section provides evidence for or against the multivariate architecture.


In [ ]:
# ── Build a joint returns DataFrame ───────────────────────────────────────────
joint_returns = pd.DataFrame({
    NAMES[t]: dfs[t]['log_return'] for t in TICKERS
}).dropna()

# ── Static correlation: training vs crisis periods ────────────────────────────
periods = {
    'Training (2015–2019)': ('2015-01-01', '2019-12-31'),
    'COVID-19 (Feb–May 2020)': ('2020-02-15', '2020-05-01'),
    'Fed Hikes (2022)': ('2022-01-01', '2022-12-31'),
}

fig, axes = plt.subplots(1, len(periods), figsize=(18, 5))

for j, (period_name, (start, end)) in enumerate(periods.items()):
    mask    = (joint_returns.index >= start) & (joint_returns.index <= end)
    corr    = joint_returns[mask].corr()

    sns.heatmap(corr, ax=axes[j], cmap='coolwarm', center=0,
                vmin=-1, vmax=1, annot=True, fmt='.2f',
                linewidths=0.5, cbar=(j == len(periods) - 1),
                xticklabels=corr.columns, yticklabels=corr.columns)
    axes[j].set_title(period_name, fontweight='bold')
    axes[j].tick_params(axis='x', rotation=30, labelsize=9)
    axes[j].tick_params(axis='y', rotation=0,  labelsize=9)

plt.suptitle('Cross-Asset Correlation Matrix — Regime Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_correlation_regime_comparison.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Rolling 30-day correlation — two key pairs ────────────────────────────────
roll_corr_cib_ec     = joint_returns['Bancolombia'].rolling(30).corr(
                            joint_returns['Ecopetrol'])
roll_corr_tgls_cmtoy = joint_returns['Tecnoglass'].rolling(30).corr(
                            joint_returns['Argos'])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=roll_corr_cib_ec.index, y=roll_corr_cib_ec,
    mode='lines', name='Bancolombia – Ecopetrol',
    line=dict(color='#1f77b4', width=1.5)
))
fig.add_trace(go.Scatter(
    x=roll_corr_tgls_cmtoy.index, y=roll_corr_tgls_cmtoy,
    mode='lines', name='Tecnoglass – Argos',
    line=dict(color='#d62728', width=1.5)
))

for crisis in CRISIS_PERIODS:
    fig.add_vrect(
        x0=crisis['start'], x1=crisis['end'],
        fillcolor=crisis['color'], layer='below', line_width=0,
        annotation_text=crisis['name'], annotation_font_size=9
    )

fig.add_hline(y=0, line_dash='dot', line_color='grey', line_width=0.8)

fig.update_layout(
    title='<b>30-Day Rolling Pearson Correlation — Selected Asset Pairs</b>',
    xaxis_title='Date',
    yaxis_title='Pearson Correlation',
    yaxis=dict(range=[-1, 1]),
    template='plotly_white',
    height=450, width=1050
)
fig.show()


In [ ]:
# ── Quantify correlation jump during crises ────────────────────────────────────
pairs = [('Bancolombia', 'Ecopetrol'), ('Tecnoglass', 'Argos'),
         ('Bancolombia', 'Tecnoglass'), ('Ecopetrol', 'Argos')]

print(f"{'Pair':<30}  {'Baseline':>10} {'COVID':>10} {'Fed Hikes':>10} {'Delta COVID':>12}")
print('-' * 78)

for (a1, a2) in pairs:
    base = joint_returns.loc['2015-01-01':'2019-12-31', [a1, a2]].corr().iloc[0, 1]
    cov  = joint_returns.loc['2020-02-15':'2020-05-01', [a1, a2]].corr().iloc[0, 1]
    fed  = joint_returns.loc['2022-01-01':'2022-12-31', [a1, a2]].corr().iloc[0, 1]
    print(f"{a1+' – '+a2:<30}  {base:>10.4f} {cov:>10.4f} {fed:>10.4f} {cov-base:>12.4f}")


### **Cross-Asset Correlation Analysis — Summary**

#### **Insight**
- The CIB–EC pair exhibits a moderate baseline correlation (~0.25–0.35) driven by shared Colombian macroeconomic exposure. During the COVID-19 crash, this correlation surges to > 0.70, confirming systemic co-movement.
- TGLS–CMTOY and other cross-sector pairs show low baseline correlation (< 0.20), confirming sector diversification under normal conditions.
- During the COVID crisis, all pairwise correlations increase substantially (correlation contagion), including pairs that are structurally unrelated (e.g., TGLS–EC). This cross-asset synchronisation is a hallmark of market anomalies and provides a powerful secondary detection signal.
- The Fed hike cycle shows a more moderate correlation elevation, consistent with a macro risk-off environment rather than a panic.

#### **Preprocessing Decision**
The **multivariate architecture** is recommended: all four assets' feature vectors are concatenated along the feature axis at each timestep, yielding an input of shape `(batch_size, 30, 28)`. This enables the encoder to represent cross-asset correlation structure.

Alternative (univariate per asset): `(batch_size, 30, 7)`. Simpler but misses correlation anomalies.

#### **Impact on Model Design**
- Multivariate input: `(B, 30, 28)` → Encoder LSTM(64) → LSTM(32) → Dense(latent_dim=16) → Decoder Dense(32) → LSTM(32) → LSTM(64) → Dense(28).
- The model can detect anomalies attributable to abnormal co-movement even when individual asset returns appear moderate.
- Reconstruction error can be decomposed per asset to localise the source of the anomaly.

#### **Risks**
- **Dimensionality:** The multivariate input (28 features) increases the parameter count significantly. Regularisation (dropout, L2) and a larger training set are required to avoid over-fitting.
- **Cross-asset contamination:** If one asset (e.g., TGLS) has an idiosyncratic anomaly not shared by others, the multivariate model may distribute reconstruction error across all assets, diluting the signal. Per-asset reconstruction error monitoring is essential.


---
## 11. EDA Summary — Architectural Design Decisions

The following table consolidates all design decisions derived from the preceding analyses. Each decision is directly traceable to quantitative evidence produced in this notebook.


In [ ]:
# ── Consolidated design decision matrix ──────────────────────────────────────
decisions = [
    ('Primary input variable',      'Log return r_t = ln(P_t/P_{t-1})',
     'ADF test: close prices non-stationary; log returns stationary (p<0.001)'),

    ('Sequence length (T)',         '30 trading days',
     'ACF on squared returns: significant lags up to ~30; financial month intuition'),

    ('Feature vector (per step)',   '[log_return, roll_vol_21, vol_zscore,\n'
                                    '  rsi_14, macd_hist, bb_pctB, bb_width] — 7 features',
     'Feature correlation matrix: features provide complementary information'),

    ('Architecture type',           'Multivariate (4 assets × 7 features = 28)',
     'Correlation contagion: crisis anomalies elevate all pairwise correlations'),

    ('Train / Val / Test split',    '2015–2019 / 2020 / 2021–present',
     'Volatility analysis: pre-2020 = low-vol baseline; post-2020 = crisis regimes'),

    ('Normalisation',               'StandardScaler fitted on training set only',
     'Temporal leakage prevention: no future statistics used'),

    ('Anomaly threshold',           '95th percentile of training-set reconstruction MSE',
     'Non-Gaussian returns: threshold must be empirical, not analytical'),

    ('Encoder architecture',        'LSTM(64) → LSTM(32) → Dense(latent_dim=16)',
     'ARCH effects suggest 2-layer LSTM to capture both short and long-memory patterns'),

    ('Loss function',               'Mean Squared Error (MSE)',
     'Reconstruction-based anomaly detection: anomaly score = mean MSE over window'),

    ('Known anomaly benchmarks',    'COVID (Feb–May 2020): all assets\n'
                                    '  Oil crisis (Jul 2015 – Feb 2016): EC\n'
                                    '  Fed hikes (2022): all assets (moderate)',
     'Rolling volatility and extreme event analysis'),
]

print("CONSOLIDATED ARCHITECTURAL DESIGN DECISIONS")
print("Derived from EDA — Colombian ADRs (2015–2026)")
print("=" * 80)
for dim, decision, evidence in decisions:
    print(f"\n  Dimension:  {dim}")
    print(f"  Decision:   {decision}")
    print(f"  Evidence:   {evidence}")
print("\n" + "=" * 80)
print("\nInput tensor shape:")
print("  Multivariate: (batch_size, 30, 28)")
print("  Per-asset:    (batch_size, 30,  7)")
print()
print("Encoder: LSTM(64) -> LSTM(32) -> Dense(16)  [bottleneck / latent space]")
print("Decoder: Dense(32) -> LSTM(32) -> LSTM(64) -> Dense(28 or 7)")
print("Anomaly criterion: MSE_t > percentile_95(MSE_train)")


---

### Reproducibility Note

All analyses in this notebook are fully reproducible. Data is downloaded programmatically from Yahoo Finance using `yfinance`. The random seed is fixed at 42 for any stochastic operations. Feature engineering computations are causal (no lookahead). All figures are saved to disk as PNG files.

**Next step:** Proceed to `02_preprocessing_and_windowing.ipynb` for scaler fitting, window generation, and final dataset preparation for LSTM Autoencoder training.
